# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 24 11:04:52 AM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671287,35.9,1473300,78.7,1473300,78.7
Vcells,1242646,9.5,8388608,64.0,1978712,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia

In [4]:
# ==============================================================================
# CONFIGURACIÓN GENERAL Y NUEVA SELECCIÓN DE 5 EXPERIMENTOS (LOTE 2)
# ==============================================================================
PARAM <- list()
PARAM$semilla_primigenia <- 200003
PARAM$num_trees_max      <- 32
PARAM$rpart$cp           <- -1

# Definimos las siguientes 5 combinaciones clave
grilla_experimentos <- data.frame(
  feature_fraction = c(0.25, 0.35, 0.30, 0.25, 0.40),
  maxdepth         = c(11,   11,   12,   10,   12),
  minsplit         = c(500,  550,  600,  400,  500),
  minbucket        = c(15,   20,   15,   10,   15)
)

# Cortes de ensamble a guardar/subir
grabar <- c(1, 2, 4, 8, 16, 32)

cat("Total de combinaciones a probar:", nrow(grilla_experimentos), "\n")
print(grilla_experimentos)

Total de combinaciones a probar: 5 
  feature_fraction maxdepth minsplit minbucket
1             0.25       11      500        15
2             0.35       11      550        20
3             0.30       12      600        15
4             0.25       10      400        10
5             0.40       12      500        15


In [5]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4020"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [6]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [7]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [8]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [9]:
# que tamanos de ensemble grabo a disco
grabar <- c(1, 2, 4, 8, 16, 32)

In [10]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
tb_prediccion[, prob_acumulada := 0]

In [11]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [12]:
# ==============================================================================
# BUCLE DE ENTRENAMIENTO Y PREDICCIÓN (CON RESUME AUTOMÁTICO Y MANEJO DE ERRORES)
# ==============================================================================
for (exp_idx in 1:nrow(grilla_experimentos)) {

  # Archivo testigo final del experimento actual
  archivo_final <- paste0(
    "KA420_exp", sprintf("%.2d", exp_idx),
    "_", sprintf("%.3d", PARAM$num_trees_max), ".csv"
  )

  # Si ya terminó este experimento en una corrida previa, lo saltea
  if (file.exists(archivo_final)) {
    message(sprintf("\n[SKIP] Exp %d/%d ya completado anteriormente. Continuando...",
                    exp_idx, nrow(grilla_experimentos)))
    next
  }

  # Asignar hiperparámetros del experimento actual
  p_ff        <- grilla_experimentos$feature_fraction[exp_idx]
  p_maxdepth  <- grilla_experimentos$maxdepth[exp_idx]
  p_minsplit  <- grilla_experimentos$minsplit[exp_idx]
  p_minbucket <- grilla_experimentos$minbucket[exp_idx]

  PARAM$rpart$maxdepth   <- p_maxdepth
  PARAM$rpart$minsplit   <- p_minsplit
  PARAM$rpart$minbucket  <- p_minbucket
  PARAM$feature_fraction <- p_ff

  # Reiniciar el acumulador de probabilidades para la corrida
  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  # Fijar semilla para reproducibilidad
  set.seed(PARAM$semilla_primigenia)

  message(sprintf("\n--- Iniciando Exp %d/%d: ff=%.2f md=%d ms=%d mb=%d ---",
                  exp_idx, nrow(grilla_experimentos), p_ff, p_maxdepth, p_minsplit, p_minbucket))

  # Generar los 32 árboles
  for (arbolito in seq(PARAM$num_trees_max)) {
    message(arbolito, " ")

    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * PARAM$feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse = " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(
      formulita,
      data = dtrain,
      xval = 0,
      control = PARAM$rpart
    )

    prediccion <- predict(modelo, dfuture, type = "prob")
    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    if (arbolito %in% grabar) {
      umbral_corte <- (1 / 40) * arbolito
      tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

      archivo_kaggle <- paste0(
        "KA420_exp", sprintf("%.2d", exp_idx),
        "_", sprintf("%.3d", arbolito), ".csv"
      )

      # Guardar siempre primero en Google Drive
      fwrite(
        tb_prediccion[, list(numero_de_cliente, Predicted)],
        file = archivo_kaggle,
        sep = ","
      )

      # Subida a Kaggle con protección ante errores
      comando <- "kaggle competitions submit"
      competencia <- "-c utn-2026-inicial"
      arch <- paste("-f", archivo_kaggle)
      mensaje <- paste0(
        "-m 'Exp=", exp_idx,
        " trees=", arbolito,
        " ff=", PARAM$feature_fraction,
        " ms=", PARAM$rpart$minsplit,
        " mb=", PARAM$rpart$minbucket,
        " md=", PARAM$rpart$maxdepth, "'"
      )

      linea <- paste(comando, competencia, arch, mensaje)

      try({
        salida <- system(linea, intern = TRUE)
        cat(salida, "\n")
      }, silent = TRUE)
    }
  }
}



--- Iniciando Exp 1/5: ff=0.25 md=11 ms=500 mb=15 ---

1 



2 



3 

4 



5 

6 

7 

8 



9 

10 

11 

12 

13 

14 

15 

16 



...

2 



3 

4 



5 

6 

7 

8 



9 

10 

11 

12 

13 

14 

15 

16 



17 

18 

19 

20 

21 

22 

23 

24 

25 

26 

27 

28 

29 

30 

31 

32 



In [13]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 24 02:18:07 PM 2026"



---

